In [1]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    evaluate \
    rouge-score \
    textstat \
    bert-score \
    sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 106.5 MB/s eta 0:00:00


In [2]:
import os
import gc
import time
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        "Total VRAM:",
        round(total_vram, 2),
        "GB"
    )

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [3]:
asset = load_dataset("facebook/asset")

print(asset)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

simplification/validation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B /  885kB            

simplification/validation-00000-of-00001(…): downloading bytes:           |  0.00B            

simplification/test-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  170kB            

simplification/test-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/359 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 359
    })
})


In [4]:
asset_df = pd.concat(
    [
        asset["validation"].to_pandas(),
        asset["test"].to_pandas()
    ],
    ignore_index=True
)

print("Unique source sentences:", len(asset_df))

print(
    "Total human references:",
    asset_df["simplifications"].apply(len).sum()
)

Unique source sentences: 2359
Total human references: 23590


In [5]:
assert len(asset_df) == 2359

assert asset_df[
    "simplifications"
].apply(len).eq(10).all()

print("✅ ASSET verification passed.")
print("Sources:", len(asset_df))

print(
    "References:",
    asset_df["simplifications"].apply(len).sum()
)

✅ ASSET verification passed.
Sources: 2359
References: 23590


In [6]:
from huggingface_hub import login

login()

In [11]:
from huggingface_hub import logout, login

logout()
login()

In [13]:
# ==========================================
# LOAD GEMMA 2 2B IT
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b-it"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("✅ Gemma loaded successfully!")
print("Model:", MODEL_NAME)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
403 Client Error. (Request ID: Root=1-6a8819bf-2c619eac038986b92a855c62;5b9e9d4e-432c-4ad7-842f-12b9e7500968)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2-2b-it to ask for access.

In [14]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6961217bc73a3e027e9a4962', 'name': 'Faizakhan11', 'fullname': 'Faiza Khan', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/6356dabac7350fd676c65fe37ec002b1.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-09-20T09:25:48.000Z'}}


In [15]:
from huggingface_hub import logout, login

logout()
login()

In [17]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6961217bc73a3e027e9a4962', 'name': 'Faizakhan11', 'fullname': 'Faiza Khan', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/6356dabac7350fd676c65fe37ec002b1.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-09-20T09:30:03.000Z'}}


In [18]:
MODEL_NAME = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("✅ Gemma loaded successfully!")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Gemma loaded successfully!


In [20]:
import evaluate
import textstat

!pip install sacremoses

sari_metric = evaluate.load("sari")

bleu_metric = evaluate.load("bleu")

rouge_metric = evaluate.load("rouge")

bertscore_metric = evaluate.load(
    "bertscore"
)

print("✅ All metrics loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 18.8 MB/s eta 0:00:00


✅ All metrics loaded successfully.


In [21]:
def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(prediction).strip()

    # =====================================
    # EMPTY PREDICTION SAFETY
    # =====================================

    if not prediction:

        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # =====================================
    # SARI
    # =====================================

    try:

        sari_result = sari_metric.compute(
            sources=[source],
            predictions=[prediction],
            references=[references]
        )

        sari = sari_result["sari"]

    except Exception:

        sari = 0.0


    # =====================================
    # BLEU
    # =====================================

    try:

        bleu_result = bleu_metric.compute(
            predictions=[prediction],
            references=[references],
            smooth=True
        )

        bleu = bleu_result["bleu"]

    except Exception:

        bleu = 0.0


    # =====================================
    # ROUGE-L
    # MEAN ACROSS ALL 10 REFERENCES
    # =====================================

    try:

        rouge_scores = []

        for reference in references:

            rouge_result = rouge_metric.compute(
                predictions=[prediction],
                references=[reference]
            )

            rouge_scores.append(
                rouge_result["rougeL"]
            )

        rouge_l = float(
            np.mean(rouge_scores)
        )

    except Exception:

        rouge_l = 0.0


    # =====================================
    # BERTScore
    # MEAN ACROSS ALL 10 REFERENCES
    # =====================================

    try:

        bert_result = bertscore_metric.compute(
            predictions=[
                prediction
            ] * len(references),

            references=references,

            lang="en"
        )

        bertscore_f1 = float(
            np.mean(
                bert_result["f1"]
            )
        )

        if np.isnan(bertscore_f1):

            bertscore_f1 = 0.0

    except Exception:

        bertscore_f1 = 0.0


    # =====================================
    # FKGL
    # =====================================

    try:

        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )

        if np.isnan(fkgl):

            fkgl = 0.0

    except Exception:

        fkgl = 0.0


    # =====================================
    # RETURN
    # =====================================

    return {

        "sari": float(sari),

        "bleu": float(bleu),

        "rougeL": float(rouge_l),

        "bertscore_f1": float(
            bertscore_f1
        ),

        "fkgl": float(fkgl)
    }

In [22]:
def cot_prompt(sentence):

    return f"""Simplify the following sentence using clear and easy-to-understand language.

Before producing the final answer, carefully reason about:
1. The main meaning of the sentence.
2. Any complex words, phrases, or structures that should be simplified.
3. Which information must be preserved to avoid changing the meaning.
4. How to make the sentence easier to understand while keeping the important information.

Do not add new information.
Do not remove important information.
Do not provide explanations in the final answer.

Return ONLY the final simplified sentence.

Sentence:
{sentence}

Final simplified sentence:"""

In [23]:
example = asset_df.iloc[0]

source = example["original"]

prompt = cot_prompt(source)

print(prompt)

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


start_time = time.perf_counter()


with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )


end_time = time.perf_counter()


inference_time = (
    end_time - start_time
)


generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]


prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()


peak_vram = (

    torch.cuda.max_memory_allocated()
    / (1024 ** 3)

    if torch.cuda.is_available()

    else None
)


print("\n" + "=" * 70)
print("SOURCE:")
print(source)

print("\n" + "=" * 70)
print("GEMMA CoT:")
print(prediction)

print(
    "\nInference time:",
    round(inference_time, 3),
    "seconds"
)

print(
    "Peak VRAM:",
    round(peak_vram, 3),
    "GB"
)

Simplify the following sentence using clear and easy-to-understand language.

Before producing the final answer, carefully reason about:
1. The main meaning of the sentence.
2. Any complex words, phrases, or structures that should be simplified.
3. Which information must be preserved to avoid changing the meaning.
4. How to make the sentence easier to understand while keeping the important information.

Do not add new information.
Do not remove important information.
Do not provide explanations in the final answer.

Return ONLY the final simplified sentence.

Sentence:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

Final simplified sentence:

SOURCE:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GEMMA CoT:
Marin, Mendocino, Lake, Napa, Solano, and Contra Costa are adja

In [24]:
test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():

    print(
        f"{metric}: {value:.4f}"
    )

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



METRICS
sari: 41.7859
bleu: 0.8432
rougeL: 0.6248
bertscore_f1: 0.9412
fkgl: 13.3727


In [25]:
# ==========================================
# GEMMA CoT CONFIGURATION
# ==========================================

COT_OUTPUT = (
    "GEMMA_COT_ASSET_2359_FINAL.csv"
)

CHECKPOINT_EVERY = 25

MAX_NEW_TOKENS = 128


if os.path.exists(COT_OUTPUT):

    cot_df = pd.read_csv(
        COT_OUTPUT
    )

    completed_ids = set(
        cot_df["sample_id"]
        .astype(int)
    )

    print(
        f"🔄 Existing CoT results: "
        f"{len(completed_ids)} completed."
    )

else:

    cot_df = pd.DataFrame()

    completed_ids = set()

    print(
        "🆕 Starting Gemma CoT "
        "from sample 1."
    )

🆕 Starting Gemma CoT from sample 1.


In [26]:
# ==========================================
# GEMMA 2B IT
# CHAIN-OF-THOUGHT
# ASSET — 2359 SAMPLES
# ==========================================


for i in range(len(asset_df)):


    # =====================================
    # RESUME
    # =====================================

    if i in completed_ids:

        continue


    # =====================================
    # DATA
    # =====================================

    example = asset_df.iloc[i]

    source = str(
        example["original"]
    )

    references = example[
        "simplifications"
    ]

    references = [
        str(ref).strip()
        for ref in references
    ]


    # =====================================
    # CoT PROMPT
    # =====================================

    prompt = cot_prompt(
        source
    )


    # =====================================
    # TOKENIZATION
    # =====================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)


    # =====================================
    # RESET VRAM
    # =====================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # GENERATION
    # =====================================

    start_time = time.perf_counter()


    try:

        with torch.no_grad():

            output_ids = model.generate(
                **inputs,

                max_new_tokens=MAX_NEW_TOKENS,

                do_sample=False,

                pad_token_id=tokenizer.eos_token_id
            )


        generated_ids = output_ids[
            0,
            inputs[
                "input_ids"
            ].shape[1]:
        ]


        prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()


    except Exception as e:

        print(
            f"\n❌ ERROR "
            f"at sample {i + 1}: {e}"
        )

        prediction = ""

        output_ids = None


    end_time = time.perf_counter()


    inference_time = (
        end_time - start_time
    )


    # =====================================
    # PEAK VRAM
    # Generation only
    # =====================================

    if torch.cuda.is_available():

        peak_vram = (

            torch.cuda.max_memory_allocated()
            / (1024 ** 3)

        )

    else:

        peak_vram = None


    # =====================================
    # METRICS
    # =====================================

    scores = evaluate_one_sample(
        source,
        prediction,
        references
    )


    # =====================================
    # SAVE RESULT
    # =====================================

    result = {

        "sample_id": i,

        "model": "Gemma-2-2B-IT",

        "prompting": "chain-of-thought",

        "original": source,

        "generated": prediction,

        "references": " ||| ".join(
            references
        ),

        "sari": scores["sari"],

        "bleu": scores["bleu"],

        "rougeL": scores["rougeL"],

        "bertscore_f1":
            scores["bertscore_f1"],

        "fkgl": scores["fkgl"],

        "inference_time_sec":
            inference_time,

        "peak_vram_gb":
            peak_vram
    }


    # =====================================
    # APPEND
    # =====================================

    cot_df = pd.concat(

        [
            cot_df,
            pd.DataFrame(
                [result]
            )
        ],

        ignore_index=True
    )


    completed_ids.add(i)


    # =====================================
    # CHECKPOINT
    # =====================================

    if (

        len(completed_ids)
        % CHECKPOINT_EVERY == 0

        or

        i == len(asset_df) - 1
    ):

        cot_df.to_csv(

            COT_OUTPUT,

            index=False
        )

        print(
            f"\n💾 CHECKPOINT: "
            f"{len(completed_ids)}"
            f"/{len(asset_df)} saved"
        )


    # =====================================
    # PROGRESS
    # =====================================

    print(

        f"[{i + 1}/"
        f"{len(asset_df)}] "

        f"SARI="
        f"{scores['sari']:.2f} | "

        f"BLEU="
        f"{scores['bleu']:.3f} | "

        f"ROUGE-L="
        f"{scores['rougeL']:.3f} | "

        f"BERT="
        f"{scores['bertscore_f1']:.3f} | "

        f"FKGL="
        f"{scores['fkgl']:.2f} | "

        f"Time="
        f"{inference_time:.2f}s | "

        f"VRAM="
        f"{peak_vram:.2f}GB"
    )


    # =====================================
    # CLEANUP
    # =====================================

    del inputs

    if output_ids is not None:

        del output_ids


    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

[1/2359] SARI=41.79 | BLEU=0.843 | ROUGE-L=0.625 | BERT=0.941 | FKGL=13.37 | Time=2.71s | VRAM=5.90GB
[2/2359] SARI=41.15 | BLEU=0.393 | ROUGE-L=0.488 | BERT=0.942 | FKGL=6.71 | Time=1.74s | VRAM=5.90GB
[3/2359] SARI=31.26 | BLEU=0.338 | ROUGE-L=0.299 | BERT=0.910 | FKGL=8.01 | Time=1.08s | VRAM=5.90GB
[4/2359] SARI=23.31 | BLEU=0.711 | ROUGE-L=0.466 | BERT=0.932 | FKGL=10.35 | Time=1.02s | VRAM=5.90GB
[5/2359] SARI=29.29 | BLEU=0.240 | ROUGE-L=0.315 | BERT=0.938 | FKGL=5.25 | Time=0.57s | VRAM=5.90GB
[6/2359] SARI=34.73 | BLEU=0.565 | ROUGE-L=0.466 | BERT=0.940 | FKGL=11.71 | Time=0.92s | VRAM=5.90GB
[7/2359] SARI=32.49 | BLEU=0.391 | ROUGE-L=0.352 | BERT=0.939 | FKGL=8.76 | Time=1.05s | VRAM=5.90GB
[8/2359] SARI=33.42 | BLEU=0.374 | ROUGE-L=0.373 | BERT=0.930 | FKGL=11.52 | Time=0.69s | VRAM=5.90GB
[9/2359] SARI=43.87 | BLEU=1.000 | ROUGE-L=0.477 | BERT=0.961 | FKGL=6.71 | Time=1.10s | VRAM=5.90GB
[10/2359] SARI=37.33 | BLEU=0.500 | ROUGE-L=0.530 | BERT=0.945 | FKGL=2.48 | Time=0.81s

In [27]:
# ============================================================
# QWEN CoT — FINAL STATISTICAL EVALUATION
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. LOAD COMPLETED CSV
# ------------------------------------------------------------

CSV_FILE = "qwen_cot_final.csv"   # <-- change if your filename differs

df = pd.read_csv(CSV_FILE)

print("=" * 70)
print("QWEN CoT — FINAL EVALUATION")
print("=" * 70)

print(f"CSV File           : {CSV_FILE}")
print(f"Total rows         : {len(df)}")


# ------------------------------------------------------------
# 2. DISPLAY COLUMNS
# ------------------------------------------------------------

print("\nColumns found:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. METRICS
# ------------------------------------------------------------

metrics = [
    "SARI",
    "BLEU",
    "ROUGE-L",
    "BERTScore F1",
    "FKGL"
]

summary = []

for metric in metrics:

    if metric not in df.columns:
        print(f"⚠️ Column not found: {metric}")
        continue

    values = pd.to_numeric(df[metric], errors="coerce").dropna()

    summary.append({
        "Metric": metric,
        "Count": len(values),
        "Mean": values.mean(),
        "Median": values.median(),
        "Min": values.min(),
        "Max": values.max(),
        "Std": values.std(),
    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# 4. INFERENCE TIME
# ------------------------------------------------------------

time_column = None

for col in ["Time", "Inference Time", "Inference_Time", "Time (s)"]:
    if col in df.columns:
        time_column = col
        break

if time_column:

    time_values = pd.to_numeric(
        df[time_column], errors="coerce"
    ).dropna()

    total_time = time_values.sum()
    mean_time = time_values.mean()
    median_time = time_values.median()
    min_time = time_values.min()
    max_time = time_values.max()
    std_time = time_values.std()

else:
    total_time = mean_time = median_time = min_time = max_time = std_time = np.nan


# ------------------------------------------------------------
# 5. VRAM
# ------------------------------------------------------------

vram_column = None

for col in ["VRAM", "VRAM (GB)", "VRAM_GB", "Peak VRAM"]:
    if col in df.columns:
        vram_column = col
        break

if vram_column:

    vram_values = pd.to_numeric(
        df[vram_column], errors="coerce"
    ).dropna()

    mean_vram = vram_values.mean()
    median_vram = vram_values.median()
    min_vram = vram_values.min()
    max_vram = vram_values.max()
    std_vram = vram_values.std()

else:
    mean_vram = median_vram = min_vram = max_vram = std_vram = np.nan


# ------------------------------------------------------------
# 6. RESPONSE SUCCESS / FAILURE
# ------------------------------------------------------------

response_column = None

for col in ["Generated", "Response", "generated", "response"]:
    if col in df.columns:
        response_column = col
        break

if response_column:

    responses = df[response_column].fillna("").astype(str)

    empty_responses = (
        responses.str.strip().eq("")
    ).sum()

    successful_responses = len(df) - empty_responses

else:
    empty_responses = np.nan
    successful_responses = np.nan


# ------------------------------------------------------------
# 7. PRINT METRIC STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METRIC STATISTICS")
print("=" * 70)

for _, row in summary_df.iterrows():

    print(f"\n{row['Metric']}")
    print(f"  Count  : {int(row['Count'])}")
    print(f"  Mean   : {row['Mean']:.4f}")
    print(f"  Median : {row['Median']:.4f}")
    print(f"  Min    : {row['Min']:.4f}")
    print(f"  Max    : {row['Max']:.4f}")
    print(f"  Std    : {row['Std']:.4f}")


# ------------------------------------------------------------
# 8. INFERENCE STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE TIME")
print("=" * 70)

print(f"Total Time   : {total_time:.2f} seconds")
print(f"Total Time   : {total_time / 60:.2f} minutes")
print(f"Total Time   : {total_time / 3600:.2f} hours")
print(f"Mean Time    : {mean_time:.4f} sec/sample")
print(f"Median Time  : {median_time:.4f} sec/sample")
print(f"Min Time     : {min_time:.4f} sec")
print(f"Max Time     : {max_time:.4f} sec")
print(f"Std Time     : {std_time:.4f} sec")


# ------------------------------------------------------------
# 9. VRAM STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VRAM")
print("=" * 70)

if not pd.isna(mean_vram):

    print(f"Mean VRAM    : {mean_vram:.2f} GB")
    print(f"Median VRAM  : {median_vram:.2f} GB")
    print(f"Min VRAM     : {min_vram:.2f} GB")
    print(f"Max VRAM     : {max_vram:.2f} GB")
    print(f"Std VRAM     : {std_vram:.2f} GB")

else:
    print("VRAM column not found.")


# ------------------------------------------------------------
# 10. RESPONSE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESPONSE STATUS")
print("=" * 70)

print(f"Total Samples        : {len(df)}")
print(f"Successful Responses : {successful_responses}")
print(f"Empty Responses      : {empty_responses}")


# ------------------------------------------------------------
# 11. SAVE METRIC SUMMARY
# ------------------------------------------------------------

summary_file = "qwen_cot_metric_summary.csv"

summary_df.to_csv(
    summary_file,
    index=False
)


# ------------------------------------------------------------
# 12. SAVE OVERALL SUMMARY
# ------------------------------------------------------------

overall_summary = pd.DataFrame([{

    "Model": "Qwen2.5-3B-Instruct",
    "Prompting": "Chain-of-Thought",
    "Samples": len(df),

    "SARI_Mean": summary_df.loc[
        summary_df["Metric"] == "SARI", "Mean"
    ].iloc[0] if "SARI" in summary_df["Metric"].values else np.nan,

    "SARI_Median": summary_df.loc[
        summary_df["Metric"] == "SARI", "Median"
    ].iloc[0] if "SARI" in summary_df["Metric"].values else np.nan,

    "BLEU_Mean": summary_df.loc[
        summary_df["Metric"] == "BLEU", "Mean"
    ].iloc[0] if "BLEU" in summary_df["Metric"].values else np.nan,

    "BLEU_Median": summary_df.loc[
        summary_df["Metric"] == "BLEU", "Median"
    ].iloc[0] if "BLEU" in summary_df["Metric"].values else np.nan,

    "ROUGE_L_Mean": summary_df.loc[
        summary_df["Metric"] == "ROUGE-L", "Mean"
    ].iloc[0] if "ROUGE-L" in summary_df["Metric"].values else np.nan,

    "ROUGE_L_Median": summary_df.loc[
        summary_df["Metric"] == "ROUGE-L", "Median"
    ].iloc[0] if "ROUGE-L" in summary_df["Metric"].values else np.nan,

    "BERTScore_Mean": summary_df.loc[
        summary_df["Metric"] == "BERTScore F1", "Mean"
    ].iloc[0] if "BERTScore F1" in summary_df["Metric"].values else np.nan,

    "BERTScore_Median": summary_df.loc[
        summary_df["Metric"] == "BERTScore F1", "Median"
    ].iloc[0] if "BERTScore F1" in summary_df["Metric"].values else np.nan,

    "FKGL_Mean": summary_df.loc[
        summary_df["Metric"] == "FKGL", "Mean"
    ].iloc[0] if "FKGL" in summary_df["Metric"].values else np.nan,

    "FKGL_Median": summary_df.loc[
        summary_df["Metric"] == "FKGL", "Median"
    ].iloc[0] if "FKGL" in summary_df["Metric"].values else np.nan,

    "Inference_Total_sec": total_time,
    "Inference_Mean_sec": mean_time,
    "Inference_Median_sec": median_time,
    "Inference_Min_sec": min_time,
    "Inference_Max_sec": max_time,

    "VRAM_Mean_GB": mean_vram,
    "VRAM_Median_GB": median_vram,
    "VRAM_Min_GB": min_vram,
    "VRAM_Max_GB": max_vram,

    "Successful": successful_responses,
    "Empty": empty_responses

}])


overall_file = "qwen_cot_final_summary.csv"

overall_summary.to_csv(
    overall_file,
    index=False
)


# ------------------------------------------------------------
# 13. FINAL DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 QWEN CoT FINAL EVALUATION COMPLETE")
print("=" * 70)

print(f"Samples       : {len(df)}")
print(f"Metric file   : {summary_file}")
print(f"Final summary : {overall_file}")

print("\n📊 FINAL SUMMARY:")
display(overall_summary)

print("\n" + "=" * 70)
print("✅ QWEN CoT IS NOW FULLY FINALIZED")
print("=" * 70)

FileNotFoundError: [Errno 2] No such file or directory: 'qwen_cot_final.csv'

In [28]:
import os

print("CSV FILES FOUND:")
print("=" * 60)

csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]

for i, f in enumerate(csv_files, 1):
    print(f"{i}. {f}")

print("=" * 60)
print(f"Total CSV files: {len(csv_files)}")

CSV FILES FOUND:
1. GEMMA_COT_ASSET_2359_FINAL.csv
Total CSV files: 1


In [31]:
# ============================================================
# GEMMA CoT — FINAL EVALUATION
# ============================================================

import pandas as pd

FILE = "GEMMA_COT_ASSET_2359_FINAL.csv"

df = pd.read_csv(FILE)

print("=" * 70)
print("🎯 GEMMA CoT FINAL EVALUATION")
print("=" * 70)

print(f"Samples: {len(df)}")

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

metrics = ["sari", "bleu", "rougeL", "bertscore_f1", "fkgl"]

for metric in metrics:

    values = pd.to_numeric(df[metric], errors="coerce").dropna()

    print("\n" + "-" * 70)
    print(metric.upper()) # Print in uppercase for display
    print("-" * 70)

    print(f"Mean       : {values.mean():.4f}")
    print(f"Median     : {values.median():.4f}")
    print(f"Minimum    : {values.min():.4f}")
    print(f"Maximum    : {values.max():.4f}")
    print(f"Std Dev    : {values.std():.4f}")


# ------------------------------------------------------------
# INFERENCE TIME
# ------------------------------------------------------------

time = pd.to_numeric(df["inference_time_sec"], errors="coerce").dropna()

print("\n" + "=" * 70)
print("⏱️ INFERENCE TIME")
print("=" * 70)

print(f"Mean       : {time.mean():.4f} sec/sample")
print(f"Median     : {time.median():.4f} sec/sample")
print(f"Minimum    : {time.min():.4f} sec")
print(f"Maximum    : {time.max():.4f} sec")
print(f"Total      : {time.sum():.2f} sec")
print(f"Total      : {time.sum()/60:.2f} min")
print(f"Total      : {time.sum()/3600:.2f} hours")


# ------------------------------------------------------------
# VRAM
# ------------------------------------------------------------

vram = pd.to_numeric(df["peak_vram_gb"], errors="coerce").dropna()

print("\n" + "=" * 70)
print("💾 VRAM")
print("=" * 70)

print(f"Mean       : {vram.mean():.2f} GB")
print(f"Median     : {vram.median():.2f} GB")
print(f"Minimum    : {vram.min():.2f} GB")
print(f"Maximum    : {vram.max():.2f} GB")


# ------------------------------------------------------------
# RESPONSE STATUS
# ------------------------------------------------------------

generated = df["generated"].fillna("").astype(str)

empty = (generated.str.strip() == "").sum()
successful = len(df) - empty

print("\n" + "=" * 70)
print("📝 RESPONSE STATUS")
print("=" * 70)

print(f"Total       : {len(df)}")
print(f"Successful  : {successful}")
print(f"Empty       : {empty}")


# ------------------------------------------------------------
# SAVE FINAL STATISTICS
# ------------------------------------------------------------

final_stats = pd.DataFrame({
    "Metric": metrics,
    "Mean": [
        df[m].mean() for m in metrics
    ],
    "Median": [
        df[m].median() for m in metrics
    ],
    "Min": [
        df[m].min() for m in metrics
    ],
    "Max": [
        df[m].max() for m in metrics
    ],
    "Std": [
        df[m].std() for m in metrics
    ]
})

final_stats.to_csv(
    "GEMMA_COT_FINAL_METRICS.csv",
    index=False
)

print("\n" + "=" * 70)
print("🎉 GEMMA CoT FINAL EVALUATION COMPLETE")
print("=" * 70)

display(final_stats)


🎯 GEMMA CoT FINAL EVALUATION
Samples: 2359

----------------------------------------------------------------------
SARI
----------------------------------------------------------------------
Mean       : 36.8143
Median     : 36.2068
Minimum    : 13.9206
Maximum    : 69.1794
Std Dev    : 9.7334

----------------------------------------------------------------------
BLEU
----------------------------------------------------------------------
Mean       : 0.5900
Median     : 0.6000
Minimum    : 0.0098
Maximum    : 1.0000
Std Dev    : 0.2628

----------------------------------------------------------------------
ROUGEL
----------------------------------------------------------------------
Mean       : 0.5098
Median     : 0.5124
Minimum    : 0.0329
Maximum    : 0.9102
Std Dev    : 0.1568

----------------------------------------------------------------------
BERTSCORE_F1
----------------------------------------------------------------------
Mean       : 0.9384
Median     : 0.9398
Minimum    

,Metric,Mean,Median,Min,Max,Std
0,sari,36.814313,36.206837,13.920619,69.179378,9.733417
1,bleu,0.589999,0.600034,0.009841,1.000000,0.262802
2,rougeL,0.509841,0.512436,0.032897,0.910225,0.156799
3,bertscore_f1,0.938413,0.939826,0.843365,0.992596,0.021785
4,fkgl,7.809811,7.600000,-2.230000,24.913333,3.812748
